In [ ]:
knitr::opts_chunk$set(echo = TRUE)
library(vroom)
library(ggplot2)
setwd("/Users/sascha/Nextcloud/R-Scripts/DPAetal_Sampling")

brg <- vroom("brg.tsv")
ph <- vroom("ph.tsv")
zwi <- vroom("zwi.tsv")

brg$Quelle <- "BRG"
ph$Quelle <- "PH"
zwi$Quelle <- "ZWI"

all <- rbind(brg, ph, zwi)

all$yr <- sapply(all$Metadaten, FUN = function (x) {
  Quelle <- substr(x, 1, 2)
  if (Quelle %in% c("BR", "ZW")) {
    as.numeric(paste0("20", substr(x, 4, 5)))
  } else {
    as.numeric(paste0("20", substr(x, 3, 4)))
  }
})

__Korpussiglen__:

* BRG: Brigitte
* PH: Psychologie Heute
* ZWI: Zeit Wissen

### Verteilung der Texte über Jahre

In [ ]:
table(all$Quelle)

### Aufgesplittet nach Quelle

In [ ]:
table(all$yr, all$Quelle)

Man sieht sehr gut die Abdeckungsunterschiede:

* BRG: 2009-2020
* PH:  2005-2011
* ZWI: 2009-2020 mit Lücken bei 2021 und 2016

### Textlängen

Ich entferne zunächst alle Texte, die mehr als 5000 Token haben. Das sind 92 Texte. Die übrigen Texte verteilen sich folgendermaßen.

In [ ]:
all2 <- all[all$Token <= 5000,]
ggplot(all2, aes(x = Quelle, y = Token, fill = Quelle)) +
  geom_violin(alpha = .5, draw_quantiles = c(1/4, 1/2, 3/4)) +
  labs(subtitle = paste0("92 Texte mit >5000 Token gelöscht, n = ",
                         nrow(all2)))

Entfernt man alle 4063 Texte, die mehr als 1250 Token haben (oberes DPA-Limit, das wir angewendet haben), verteilen sich die Texte wie folgt.

In [ ]:
all3 <- all[all$Token <= 1250,]

ggplot(all3, aes(x = Quelle, y = Token, fill = Quelle)) +
  geom_violin(alpha = .5, draw_quantiles = c(1/4, 1/2, 3/4)) +
  labs(subtitle = paste0("4063 Texte mit >1250 Token gelöscht, n = ",
                         nrow(all3)))

Nun kann man sich anschauen, wie die letzte Selektion (<= 1250 Token) auf die unterschiedlichen Quellen "wirkt". Die Angaben sind in Prozent der Texte für die jeweilige Quelle.

In [ ]:
tab <- prop.table(table(all$Quelle, all$Token <= 1250), margin = 1) * 100
colnames(tab) <- c("ausgeschlossen", "eingeschlossen")
round(tab, 1)

Wir sehen, dass für Zeit Wissen prozentual mehr Texte ausgeschlossen werden. In den Violinen-Plots oben hatte sich ja bereits angedeutet, dass diese Texte tendentiell länger sind. Ich würde daher dafür plädieren, perzentilbasierte Grenzwerte pro Quelle festzulegen - zumindest als obere Grenze. So würden in jeder Quelle ungefähr gleich viel Prozent der Texte ausgeschlossen.

Im folgenden Plot sehen wir die empirischen kumulativen Verteilungsfunktionen (EKVF) für die drei Quellen (basierend auf dem Datensatz ohne jene Texte, die länger sind als 5000 Token). Dort kann man sehen, welche Grenze man pro Quelle ansetzen müsste, um (zum Beispiel) bei 10 % Ausschluss vom oberen Ende zu landen. Das ist jeweils der Schnittpunkt der gestrichelten Linie mit der jeweiligen EKVF für die Quellen.

In [ ]:
# Emp.cum.dist.function für Quellen
ecdf.brg <- ecdf(all2[all2$Quelle == "BRG",]$Token)
ecdf.zwi <- ecdf(all2[all2$Quelle == "ZWI",]$Token)
ecdf.ph <- ecdf(all2[all2$Quelle == "PH",]$Token)

curve(ecdf.brg, from = 1, to = 3000, col = "red", lwd = 2,
      bty = "n", xlab = "Grenze maximale Tokenanzahl", y = "Anteil eingeschlossener Texte (1 = 100%)")
curve(ecdf.zwi, from = 1, to = 3000, add = T, col = "blue", lwd = 2)
curve(ecdf.ph, from = 1, to = 3000, add = T, col = "green", lwd = 2)
abline(h = .9, lty = "dashed")
legend(col = c("red", "green", "blue", "black"),
       legend = c("BRG", "PH", "ZWI", "90%-Grenze"),
       x = "bottomright", lty = c(1, 1, 1, 2), lwd = c(2, 2, 2, 1),
       inset = .05)

Die Grenzen für 90% wären wie folgt:

In [ ]:
round(c(BRG = quantile(all2[all2$Quelle == "BRG",]$Token, .9), PH = quantile(all2[all2$Quelle == "PH",]$Token, .9), ZWI = quantile(all2[all2$Quelle == "ZWI",]$Token, .9)))

Wenden wir diese Grenzen an und nehmen eine Mindestlänge von 250 Token an (wie für DPA-Texte), verteilen sich die Texte wie folgt auf Quellen und Jahre.

In [ ]:
all.sel <- subset(all, ((Quelle == "BRG" & Token <= 1753) |
                    (Quelle == "PH" & Token <= 2196) |
                    (Quelle == "ZWI" & Token <= 2636)) & Token >= 250)
table(all.sel$yr, all.sel$Quelle)

### Text-Sampling

Wir haben jetzt alle "Kandidaten-Texte" für das Sampling mit angepasster oberer Grenze. Ich gehe davon aus, dass jede der Quellen im Sample gleich vertreten sein soll. Wir können also nicht einfach aus allen Texten ohne Ansehen der Quelle sampeln, da es viel mehr Brigitte-Texte gibt als andere. Wenn ich bspw. aus jeder Quelle 100 Texte sample, könnte eine Längenverteilung so aussehen. Für jedes Sampling ändert sich das natürlich leicht.

In [ ]:
library(dplyr)
all.sel %>% group_by(Quelle) %>%
  slice_sample(n = 100) -> smp.txt
ggplot(smp.txt, aes(x = Quelle, y = Token, fill = Quelle)) +
  geom_violin(alpha = .5, draw_quantiles = c(1/4, 1/2, 3/4)) +
  labs(subtitle = "100 Texte pro Quelle, mindestens 250 Token, angepasste Obergrenze")

### Vergleich mit bisher gesampelten DPA-Texten

In [ ]:
ch12.files <- dir("/Users/sascha/Nextcloud/Daten/Annotationschargen_EmpGL",
                  pattern = "^(Charge1|Charge2)_(Anno1|Anno2|beide)", full.names = T)
ch12.files <- do.call("rbind", lapply(ch12.files, vroom))

Nun können wir uns noch die Verteilung der neuen Daten im Vergleich zu den bisher gesampelten DPA-Texten anschauen.

Für Länge:

In [ ]:
dpa <- ch12.files[,c("year", "tok")]
names(dpa) <- c("yr", "Token")
dpa$Quelle <- "_DPA"
neu.dpa <- rbind(smp.txt[,c("yr", "Token", "Quelle")], dpa)
ggplot(neu.dpa, aes(x = Quelle, y = Token, fill = Quelle)) +
  geom_violin(alpha = .5, draw_quantiles = c(1/4, 1/2, 3/4)) +
  labs(subtitle = "400 Texte DPA, 100 Texte pro BRG/PH/ZWI")

Für Jahre:

In [ ]:
table(neu.dpa$yr, neu.dpa$Quelle)

Wie zu erwarten war, sind die Texte aus den neuen Quellen tendentiell etwas länger. Ich sehe das aber nicht als Nachteil (außer insofern dass die Annotation länger dauert), denn das bildet nur die Textlängen-Verteilung in den jeweiligen Quellen ab. Ich würde es im Gegenteil eher als "künstlich" sehen, wenn man die Texte aus den neuen Quellen streng so selektieren würde, dass sie alle im Längenbereich der DPA-Texte liegen. U.a. hier würde mich Eure Meinung interessieren.

Bei den Jahren würde 2005 hinzukommen (was ich für unproblematisch halte), und die drei neuen Quellen decken unterschiedliche Jahresabschnitte ab (s. oben).

### Neu: Sampling aus den mittleren 80% der Daten pro Quelle

Wir werfen von __allen__ Texten die 10% kürzesten und 10% längsten Texte raus. Die Grenzen sind wie folgt.

In [ ]:
inner80 <- rbind(quantile(brg$Token, c(0.1, 0.9)),
                 quantile(ph$Token, c(0.1, 0.9)),
                 quantile(zwi$Token, c(0.1, 0.9)))
rownames(inner80) <- c("BRG", "PH", "ZWI")
inner80

Nach dieser Selektion sollten logischerweise von allen Texten ca. 20% rausfliegen. Besser überprüfen.

In [ ]:
all.sel.i80 <- all[(all$Token > inner80["BRG", "10%"] & all$Token < inner80["BRG", "90%"] & all$Quelle == "BRG") |
                     (all$Token > inner80["PH", "10%"] & all$Token < inner80["PH", "90%"] & all$Quelle == "PH") |
                     (all$Token > inner80["ZWI", "10%"] & all$Token < inner80["ZWI", "90%"] & all$Quelle == "ZWI"), ]
(nrow(all) - nrow(all.sel.i80)) / nrow(all) * 100

OK, hat geklappt. Wie sieht die Längenverteilung der Kandidatentexte (= Texte, aus denen gesampelt wird) aus?

In [ ]:
ggplot(all.sel.i80, aes(x = Quelle, y = Token, fill = Quelle)) +
  geom_violin(alpha = .5, draw_quantiles = c(1/4, 1/2, 3/4)) +
  labs(subtitle = "Mittlere 80% jeder Quelle")

Ich probiere jetzt 500 Samples mit 90 Texten aus, um zu sehen, wie lang diese Samples so werden im Vergleich zu den DPA-Texten. Das wären dann insgesamt 270 neu zu annotierende Texte.

In [ ]:
sample100.lens <- sapply(1:500, FUN = function (x) {
  all.sel.i80 %>% group_by(Quelle) %>%
    slice_sample(n = 90) -> samp
  sum(samp$Token)
})
summary(sum(dpa$Token) - sample100.lens)

In der Tabelle oben geben `Min.` und `Max.` an, wie viel kürzer als die DPA-Texte die Samples minimal und maximal sind. Negative Zahlen bedeuten, dass die neu gesampelnden Texte länger sind. 90 pro Quelle scheint mir OK zu sein. Ich erstelle ein entsprechendes Sample. Wenn wir davon dann je 10 Texte doppelt annotieren lassen, werden 240 Texte nur von einer Annotatorin bearbeitet. Jede Annotatorin bekommt somit 120 (nur sie) + 30 (beide) = 150 neue Texte. 

In [ ]:
# Dieser Chunk ist nur für das tatsächliche Sampling, deshalb ist eval auf F gesetzt
n.src <- 90 # Anzahl Texte pro neuer Quelle
all.sel.i80 %>% group_by(Quelle) %>%
  slice_sample(n = n.src) -> sampled.komplett
# Texte, die beide annotieren
sampled.komplett %>% group_by(Quelle) %>%
  slice_sample(n = 10) -> sample_beide
# Texte, die nur von einer annotiert werden
sample_einzeln <- sampled.komplett[!(sampled.komplett$Metadaten %in% sample_beide$Metadaten),]
# Texte, die nur Annotatorin 1 sieht
sample_einzeln %>% group_by(Quelle) %>%
  slice_sample(n = 40) -> sample_an1
# Texte, die nur Annotatorin 2 sieht
sample_an2 <- sample_einzeln[!(sample_einzeln$Metadaten %in% sample_an1$Metadaten),]
# Testen, ob alle Siglen, die An1 und An2 sehen, in sample_einzeln sind
all(c(sample_an1$Metadaten, sample_an2$Metadaten) %in% sample_einzeln$Metadaten)
# Testen, wie viele Siglen, die An1 und An2 einzeln sehen, NICHT in sample_beide sind (sollten alle sein)
all(!(c(sample_an1$Metadaten, sample_an2$Metadaten) %in% sample_beide$Metadaten))
# Testen, wie viele Texte insgesamt annotiert werden (sollten 270 sein)
nrow(sample_an1) + nrow(sample_an2) + nrow(sample_beide)
# Testen, wie viele Texte eine Annotatorin sieht (sollten 150 sein)
nrow(sample_an1) + nrow(sample_beide)
# Wie viele Siglen werden insgesamt annotiert (sollten 270 sein)
length(unique(c(sample_an1$Metadaten, sample_an2$Metadaten, sample_beide$Metadaten)))

# Rausschreiben
fwrite(sample_beide, file = "/Users/sascha/Nextcloud/Daten/Annotationschargen_EmpGL/neueTexte_Charge1_beide.csv", sep = "\t")
fwrite(sample_an1, file = "/Users/sascha/Nextcloud/Daten/Annotationschargen_EmpGL/neueTexte_Charge1_Anno1.csv", sep = "\t")
fwrite(sample_an2, file = "/Users/sascha/Nextcloud/Daten/Annotationschargen_EmpGL/neueTexte_Charge1_Anno2.csv", sep = "\t")

# Ausgabe für Jan
write(sample_beide$Metadaten, file = "/Users/sascha/Nextcloud/Daten/Annotationschargen_EmpGL/neueTexte_Charge1_Siglen_beide.txt",
      sep = "\n")
write(sample_an1$Metadaten, file = "/Users/sascha/Nextcloud/Daten/Annotationschargen_EmpGL/neueTexte_Charge1_Siglen_Anno1.txt",
      sep = "\n")
write(sample_an2$Metadaten, file = "/Users/sascha/Nextcloud/Daten/Annotationschargen_EmpGL/neueTexte_Charge1_Siglen_Anno2.txt",
      sep = "\n")

### Tatsächliche Länge des Samples

Die Gesamttoken-Anzahl des tatsächlich gezogenen Samples beträgt...

In [ ]:
beide <- vroom("/Users/sascha/Nextcloud/Daten/Annotationschargen_EmpGL/neueTexte_Charge1_beide.csv")
an1 <- vroom("/Users/sascha/Nextcloud/Daten/Annotationschargen_EmpGL/neueTexte_Charge1_Anno1.csv")
an2 <- vroom("/Users/sascha/Nextcloud/Daten/Annotationschargen_EmpGL/neueTexte_Charge1_Anno2.csv")
new.tok <- sum(c(beide$Token, an1$Token, an2$Token))
new.tok

Damit ist es `r as.integer(sum(dpa$Token) - new.tok)` Token kürzer als die DPA-Texte, was m.E. OK ist.

